# OpenAI vs Anthropic vs Gemini: Same Concepts, Different Contracts

Run the same tasks through OpenAI, Anthropic and Google Gemini to see that most of what you learned in OpenAI Features & Functionality maps straight across, and where the providers genuinely differ.

[Claude Console](https://platform.claude.com/) · [OpenAI Playground](https://platform.openai.com/chat/edit) · [Google AI Studio](https://aistudio.google.com/)

Instructions, conversations, streaming, structured outputs, tools and reasoning were concepts you had to learn once. Every major lab supports them. What changes is the *contract*: parameter names, message shapes and where the state lives. Learn the concept once, then map it.

The second half covers what does **not** map neatly: prompt caching, and the fact that each lab's models are good at different things.

## Before you start

The **Claude Console** is Anthropic's equivalent of the OpenAI Playground. Use its Workbench to try a prompt, then **Get code** to copy the matching Messages API call. API keys, usage and cost live there too. Google AI Studio plays the same role for Gemini.

You need three API keys: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY` and `GEMINI_API_KEY`. Each request is billed by its provider; the calls here are small. Set `RUN_GEMINI = False` if you only have the first two keys.

Each example shows the provider SDK method and its arguments directly, with response fields read inline.

In [ ]:
# Pylint treats notebook cells as a script and pip magics as pass statements.
# pylint: disable=missing-module-docstring,wrong-import-position,wrong-import-order,unnecessary-pass
%pip install openai anthropic "google-genai<3" pandas --upgrade --quiet

In [ ]:
# pylint: disable=missing-module-docstring,invalid-name
OPENAI_MODEL = "gpt-6-luna"  # OpenAI's efficient model
ANTHROPIC_MODEL = "claude-opus-5-5"  # Anthropic's current Opus
GEMINI_MODEL = "gemini-3.8-flash"  # Google's current Flash model
RUN_GEMINI = True
RUN_CACHING = True

In [ ]:
import json
import os
from getpass import getpass
from typing import cast

from anthropic import Anthropic
from anthropic.types import MessageParam, TextBlockParam, ToolParam, ToolResultBlockParam
from google import genai
from google.genai import types
import pandas as pd
from IPython.display import display
from openai import OpenAI
from openai.types.responses import FunctionToolParam, ResponseInputParam

for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"] + (
    ["GEMINI_API_KEY"] if RUN_GEMINI else []
):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"Enter your {key}: ")

openai_client = OpenAI()
claude = Anthropic()
gemini: genai.Client | None = None
if RUN_GEMINI:
    gemini = genai.Client()

---
# Part 1: Learn it once, it maps

## 1. Instructions and input

Every provider separates *how to behave* from *what to answer*. The names differ:

| | OpenAI Responses | Anthropic Messages | Gemini |
|---|---|---|---|
| Instructions | `instructions` | `system` | `system_instruction` |
| Input | `input` | `messages` | `contents` |
| Text out | `response.output_text` | text blocks in `message.content` | `response.text` |

Claude also requires `max_tokens` on every request.

In [ ]:
INSTRUCTIONS = "You are a friendly tutor. Answer in one sentence."
QUESTION = "What is a context window?"

o = openai_client.responses.create(
    model=OPENAI_MODEL, instructions=INSTRUCTIONS, input=QUESTION
)
print("OpenAI:   ", o.output_text)

a = claude.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=2000,
    system=INSTRUCTIONS,
    messages=[{"role": "user", "content": QUESTION}],
)
print("Anthropic:", "".join(block.text for block in a.content if block.type == "text"))

if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    g = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=QUESTION,
        config=types.GenerateContentConfig(system_instruction=INSTRUCTIONS),
    )
    print("Gemini:   ", g.text)

## 2. Multi-turn conversations

Same idea, different ownership of the state. OpenAI can store the conversation and continue from `previous_response_id`. Claude's Messages API and Gemini's `generate_content` are stateless: you send the whole history every time. (You can also manage history manually with OpenAI, as you did with `store=False`.)

Keep the assistant's full content when you append it, not just the text. Claude may return thinking blocks alongside the text.

In [ ]:
FIRST = "My name is Sam. Reply with one word."
SECOND = "What is my name?"

o1 = openai_client.responses.create(model=OPENAI_MODEL, input=FIRST)
o2 = openai_client.responses.create(
    model=OPENAI_MODEL, previous_response_id=o1.id, input=SECOND
)
print("OpenAI:   ", o2.output_text)

history: list[MessageParam] = [{"role": "user", "content": FIRST}]
a1 = claude.messages.create(model=ANTHROPIC_MODEL, max_tokens=2000, messages=history)
history += [
    {
        "role": "assistant",
        "content": a1.content,
    },
    {"role": "user", "content": SECOND},
]
a2 = claude.messages.create(model=ANTHROPIC_MODEL, max_tokens=2000, messages=history)
print("Anthropic:", "".join(block.text for block in a2.content if block.type == "text"))

if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    turns: list[types.ContentUnionDict] = [
        types.Content(role="user", parts=[types.Part(text=FIRST)])
    ]
    g1 = gemini.models.generate_content(model=GEMINI_MODEL, contents=turns)
    assert g1.candidates, "Gemini returned no conversation candidates"
    model_turn = g1.candidates[0].content
    assert model_turn is not None, "Gemini returned no conversation content"
    turns += [
        model_turn,  # Gemini calls the assistant role "model"
        types.Content(role="user", parts=[types.Part(text=SECOND)]),
    ]
    print(
        "Gemini:   ",
        gemini.models.generate_content(model=GEMINI_MODEL, contents=turns).text,
    )

## 3. Streaming

All three stream tokens as they are generated. OpenAI sends typed events; Anthropic's SDK gives a `text_stream` helper and the final message (with usage) at the end; Gemini yields chunks.

In [ ]:
PROMPT = "Count from 1 to 5, one number per line."

print("OpenAI:")
for event in openai_client.responses.create(
    model=OPENAI_MODEL, input=PROMPT, stream=True
):
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print("\n\nAnthropic:")
with claude.messages.stream(
    model=ANTHROPIC_MODEL,
    max_tokens=2000,
    messages=[{"role": "user", "content": PROMPT}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    final = stream.get_final_message()
print("\n(stop reason:", final.stop_reason, ")")

if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    print("\nGemini:")
    for chunk in gemini.models.generate_content_stream(
        model=GEMINI_MODEL, contents=PROMPT
    ):
        print(chunk.text or "", end="", flush=True)

## 4. Structured outputs

One JSON Schema, three envelopes. The schema itself is reused unchanged, which is the point: your application owns the schema, each provider just wants it in a different place.

In [ ]:
review_schema = {
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
        "reason": {"type": "string"},
    },
    "required": ["sentiment", "reason"],
    "additionalProperties": False,
}
REVIEW = "Classify this review: The mug is lovely but delivery took three weeks."

o = openai_client.responses.create(
    model=OPENAI_MODEL,
    input=REVIEW,
    text={
        "format": {
            "type": "json_schema",
            "name": "review",
            "strict": True,
            "schema": review_schema,
        }
    },
)
results = {"OpenAI": json.loads(o.output_text)}

a = claude.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=2000,
    messages=[{"role": "user", "content": REVIEW}],
    output_config={"format": {"type": "json_schema", "schema": review_schema}},
)
results["Anthropic"] = json.loads(
    "".join(block.text for block in a.content if block.type == "text")
)

if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    g = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=REVIEW,
        config=types.GenerateContentConfig(
            response_mime_type="application/json", response_json_schema=review_schema
        ),
    )
    assert g.text is not None, "Gemini returned no structured JSON text"
    results["Gemini"] = json.loads(g.text)

display(pd.DataFrame(results).T)

## 5. Tool calling

The same Python function and the same parameter schema, wrapped three ways:

- **OpenAI**: the model returns a `function_call` item; you reply with a `function_call_output` item.
- **Anthropic**: the model returns a `tool_use` block (`stop_reason == "tool_use"`); you reply with a `tool_result` block inside a **user** message.
- **Gemini**: the SDK can call a plain Python function for you (automatic function calling).

In [ ]:
SHIPPING_GBP = {"UK": 3, "US": 9, "FR": 6}


def get_shipping_cost(country: str) -> dict:
    """Return the shipping cost in GBP for a two-letter country code."""
    return {"country": country, "cost_gbp": SHIPPING_GBP.get(country.upper())}


DESCRIPTION = "Get the shipping cost in GBP for a two-letter country code."
PARAMETERS = {
    "type": "object",
    "properties": {"country": {"type": "string"}},
    "required": ["country"],
    "additionalProperties": False,
}
ASK = "How much is shipping to the US?"

# OpenAI: tool definition uses "parameters"
openai_tool: FunctionToolParam = {
    "type": "function",
    "name": "get_shipping_cost",
    "description": DESCRIPTION,
    "parameters": PARAMETERS,
    "strict": True,
}
inputs: ResponseInputParam = [{"role": "user", "content": ASK}]
o = openai_client.responses.create(
    model=OPENAI_MODEL, input=inputs, tools=[openai_tool]
)
inputs.extend(
    cast(ResponseInputParam, [item.model_dump(exclude_none=True) for item in o.output])
)
for item in o.output:
    if item.type == "function_call":
        result = get_shipping_cost(**json.loads(item.arguments))
        inputs.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps(result),
            }
        )
o = openai_client.responses.create(
    model=OPENAI_MODEL, input=inputs, tools=[openai_tool]
)
print("OpenAI:   ", o.output_text)

# Anthropic: tool definition uses "input_schema"
claude_tool: ToolParam = {
    "name": "get_shipping_cost",
    "description": DESCRIPTION,
    "input_schema": PARAMETERS,
}
messages: list[MessageParam] = [{"role": "user", "content": ASK}]
a = claude.messages.create(
    model=ANTHROPIC_MODEL, max_tokens=2000, tools=[claude_tool], messages=messages
)
if a.stop_reason == "tool_use":
    messages.append(
        {
            "role": "assistant",
            "content": a.content,
        }
    )
    tool_results: list[ToolResultBlockParam] = []
    for block in a.content:
        if block.type == "tool_use":
            tool_country = block.input.get("country")
            if not isinstance(tool_country, str):
                raise ValueError("get_shipping_cost requires a string country code")
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(get_shipping_cost(tool_country)),
                }
            )
    messages.append({"role": "user", "content": tool_results})
    a = claude.messages.create(
        model=ANTHROPIC_MODEL, max_tokens=2000, tools=[claude_tool], messages=messages
    )
print("Anthropic:", "".join(block.text for block in a.content if block.type == "text"))

# Gemini: pass the Python function itself
if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    g = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=ASK,
        config=types.GenerateContentConfig(tools=[get_shipping_cost]),
    )
    print("Gemini:   ", g.text)

## 6. Reasoning effort

All three let the model think before answering, and all three expose a dial for how hard it thinks. Lower effort is cheaper and faster; raise it for hard problems.

| OpenAI | Anthropic | Gemini |
|---|---|---|
| `reasoning={"effort": ...}` | `thinking={"type": "adaptive"}` + `output_config={"effort": ...}` | `thinking_config=ThinkingConfig(thinking_level=...)` |

In [ ]:
PUZZLE = (
    "A shop has 8 mugs, sells 3, then receives a delivery of 5. "
    "How many mugs now? Answer with a number."
)

o = openai_client.responses.create(
    model=OPENAI_MODEL, input=PUZZLE, reasoning={"effort": "low"}
)
assert o.usage is not None, "OpenAI returned no token usage"
print(
    "OpenAI:   ",
    o.output_text,
    "| reasoning tokens:",
    o.usage.output_tokens_details.reasoning_tokens,
)

a = claude.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=4000,
    messages=[{"role": "user", "content": PUZZLE}],
    thinking={"type": "adaptive"},
    output_config={"effort": "low"},
)
print(
    "Anthropic:",
    "".join(block.text for block in a.content if block.type == "text"),
    "| blocks:",
    [b.type for b in a.content],
)

if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    g = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=PUZZLE,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=types.ThinkingLevel.LOW)
        ),
    )
    assert g.usage_metadata is not None, "Gemini returned no token usage"
    print(
        "Gemini:   ", g.text, "| thought tokens:", g.usage_metadata.thoughts_token_count
    )

## The map so far

| Concept | OpenAI Responses | Anthropic Messages | Gemini `generate_content` |
|---|---|---|---|
| Call | `responses.create()` | `messages.create()` | `models.generate_content()` |
| Instructions | `instructions` | `system` | `system_instruction` |
| Conversation state | Server-side via `previous_response_id`, or manual | Manual: resend `messages` | Manual: resend `contents` |
| Streaming | `stream=True` events | `messages.stream()` | `generate_content_stream()` |
| JSON schema | `text.format` | `output_config.format` | `response_json_schema` |
| Tool call / result | `function_call` / `function_call_output` | `tool_use` / `tool_result` | `function_call` / `function_response` (or automatic) |
| Reasoning dial | `reasoning.effort` | `output_config.effort` | `thinking_level` |
| Token usage | `usage.input_tokens` / `output_tokens` | `usage.input_tokens` / `output_tokens` | `usage_metadata` |

If you understood the concept in the OpenAI section, you already understand it here. A thin adapter per provider is usually all the code you need.

---
# Part 2: Where they genuinely differ

## 7. Prompt caching

Caching reuses a long, unchanged prompt prefix so repeat requests are cheaper and faster. All three cache, but they hand you very different amounts of control:

| | OpenAI | Anthropic | Gemini |
|---|---|---|---|
| Default | **Automatic**, no code needed | Opt-in with `cache_control` | **Implicit** caching on by default |
| Control | Newer models also accept explicit breakpoints | You choose where breakpoints go (up to 4) and the TTL: 5 minutes or 1 hour | Create explicit cache objects with your own TTL |
| Pricing shape | Reads discounted; newer models charge for writes | Writes 1.25x (5 min) or 2x (1 h); reads about 0.1x | Reads discounted; explicit caches add storage per hour |
| Check it in | `usage.input_tokens_details.cached_tokens` | `cache_creation_input_tokens` / `cache_read_input_tokens` | `usage_metadata.cached_content_token_count` |

Below, the same long policy document is sent twice with different questions. The prefix must be long enough to qualify (each provider has a minimum, a few thousand tokens at most), so the policy is deliberately long. Don't pad real prompts just to get cache hits.

In [ ]:
if RUN_CACHING:
    POLICY = "You answer questions about our shop policy.\n" + "\n".join(
        f"Clause {i}: Orders to region {i % 40} ship within {2 + i % 5} working days; "
        f"returns are accepted for {14 + i % 3 * 7} days if the item is unused."
        for i in range(400)
    )
    QUESTIONS = [
        "How long do returns last for clause 7?",
        "When does an order to region 12 ship?",
    ]
    rows = []

    # OpenAI: nothing to configure, caching just happens
    for q in QUESTIONS:
        o = openai_client.responses.create(
            model=OPENAI_MODEL, instructions=POLICY, input=q
        )
        assert o.usage is not None, "OpenAI returned no token usage"
        rows.append(
            {
                "provider": "OpenAI",
                "question": q,
                "input": o.usage.input_tokens,
                "cache write": o.usage.input_tokens_details.cache_write_tokens,
                "cache read": o.usage.input_tokens_details.cached_tokens,
            }
        )

    # Anthropic: mark the end of the reusable prefix and pick a TTL
    cached_system: list[TextBlockParam] = [
        {
            "type": "text",
            "text": POLICY,
            "cache_control": {"type": "ephemeral", "ttl": "1h"},
        }
    ]
    for q in QUESTIONS:
        a = claude.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=2000,
            system=cached_system,
            messages=[{"role": "user", "content": q}],
        )
        rows.append(
            {
                "provider": "Anthropic",
                "question": q,
                "input": a.usage.input_tokens,
                "cache write": a.usage.cache_creation_input_tokens,
                "cache read": a.usage.cache_read_input_tokens,
            }
        )

    # Gemini: create an explicit cache object with its own TTL, then reference it
    if RUN_GEMINI:
        assert gemini is not None, "Create the Gemini client in the setup cell first"
        cache = gemini.caches.create(
            model=GEMINI_MODEL,
            config=types.CreateCachedContentConfig(
                system_instruction=POLICY, ttl="300s"
            ),
        )
        assert cache.name is not None, "Gemini returned no cache name"
        for q in QUESTIONS:
            g = gemini.models.generate_content(
                model=GEMINI_MODEL,
                contents=q,
                config=types.GenerateContentConfig(cached_content=cache.name),
            )
            assert g.usage_metadata is not None, "Gemini returned no token usage"
            rows.append(
                {
                    "provider": "Gemini",
                    "question": q,
                    "input": g.usage_metadata.prompt_token_count,
                    "cache write": None,
                    "cache read": g.usage_metadata.cached_content_token_count,
                }
            )
        gemini.caches.delete(
            name=cache.name
        )  # explicit caches bill storage until they expire

    display(pd.DataFrame(rows))

**Reading the table.** OpenAI caches without being asked; the second request usually shows cached tokens, but a hit is not guaranteed. Anthropic writes the cache on the first request and reads it on the second, exactly where you put the breakpoint and for as long as you chose. For long-lived prefixes (an agent's system prompt and tools), that control matters: you decide what is cached, for how long, and what you pay for it. Anthropic also offers a top-level `cache_control` that places the breakpoint automatically when you don't need fine-grained control.

## 8. The models are genuinely different

The APIs converge; the models do not. Each lab trains for different strengths, and their limits differ too. Start by asking each API what its model can take in:

In [ ]:
limits = [
    {
        "provider": "Anthropic",
        "model": ANTHROPIC_MODEL,
        "max input tokens": claude.models.retrieve(ANTHROPIC_MODEL).max_input_tokens,
    }
]
if RUN_GEMINI:
    assert gemini is not None, "Create the Gemini client in the setup cell first"
    gm = gemini.models.get(model=GEMINI_MODEL)
    limits.append(
        {
            "provider": "Gemini",
            "model": GEMINI_MODEL,
            "max input tokens": gm.input_token_limit,
        }
    )
display(pd.DataFrame(limits))

print(
    "OpenAI models on this key:",
    sorted(m.id for m in openai_client.models.list() if m.id.startswith("gpt"))[:10],
)

At the time of recording, a fair rule of thumb:

| Reach for | When the task is | Why |
|---|---|---|
| **Gemini** | Long documents, long video or audio | Very large context and native video understanding (see the Gemini video lecture) |
| **Claude Opus** | Coding and agentic work | Strong at reading, changing and testing real codebases |
| **Codex** (OpenAI's coding agent) | Long-running engineering tasks | Built to work through a task for a long time on its own |
| **Small tiers** (GPT Luna, Claude Haiku, Gemini Flash-Lite) | High-volume, well-defined jobs | Classification and extraction rarely need the frontier |

These shift with every release. Treat the table as a starting point and let your own evals decide.

## 9. Agents: the same idea at a higher level

Each lab now offers more than a single request. You have already met OpenAI's agents. Anthropic's equivalents:

| | OpenAI | Anthropic |
|---|---|---|
| You write the loop | Responses API + tools | Messages API + tools (or the SDK's tool runner) |
| Hosted agent with a sandbox | Agents API ([managed agents notebook](../openai_features_and_functionality/managed_agents_api.ipynb)) | Claude Managed Agents |
| A coding agent as a product or library | Codex | Claude Code and the Claude Agent SDK |

The concepts carry over: a model, instructions, tools, a loop, and somewhere for state to live. Next we build one with Claude.

## Summary

- Instructions, conversations, streaming, structured outputs, tools and reasoning are the same concepts at every lab. Learn them once.
- The contracts differ: `instructions` vs `system` vs `system_instruction`, server-side vs manual history, `function_call` vs `tool_use`. A thin adapter per provider handles it.
- Prompt caching is where control differs most: OpenAI caches automatically, Anthropic lets you choose the breakpoints, TTL and cost trade-off, Gemini offers implicit caching plus explicit cache objects.
- The models are genuinely different. Gemini for long context and video, Claude Opus for coding, Codex for long-running tasks, small tiers for volume. They all have their place.
- Pick per task, and check with your own evals rather than a leaderboard.